# PICKO Research · NB2 — **Depth**: is parameter extraction harder with more parameters?

40 full-schema tools can't all be offered at once (token limit), and a single model gives one
unreplicated score per tool. Instead we **repeatedly sample a small, bucket-balanced set** (tools from
every param-count bucket, sized to fit the encoder), finetune, and measure argument extraction — over
several iterations — so each bucket gets many measurements and we can show **error bars**.

## 0 · Colab quick-start (GPU) — run & forget, restart-safe

**On Colab: Runtime → Change runtime type → GPU (T4) first.** This cell clones the repo, pins the exact
JAX/Flax, mounts Drive (so checkpoints survive a restart), and sets the output dir. **Running locally?**
It's a no-op — just skip to cell 1.

In [ ]:
# --- Colab bootstrap (safe to re-run; no-op locally) ---
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("/content/picko"):
        !git clone -b hadar-work https://github.com/HadarBit/picko.git /content/picko
    %pip install -q "jax[cuda12]==0.10.2" "jaxlib==0.10.2" "flax==0.12.8"
    sys.path.insert(0, "/content/picko")
    from google.colab import drive; drive.mount("/content/drive")
    os.environ["PICKO_OUT_DIR"] = "/content/drive/MyDrive/picko_out"; os.makedirs(os.environ["PICKO_OUT_DIR"], exist_ok=True)
    import shutil
    src, dst = "/content/drive/MyDrive/picko_balanced.jsonl", "/content/picko/data/picko_balanced.jsonl"
    if os.path.exists(src) and not os.path.exists(dst): shutil.copy(src, dst)
    print("GPU:")
    !nvidia-smi -L
    assert os.path.exists(dst), "Data missing: it ships in the repo clone; if absent, upload picko_balanced.jsonl to /content/picko/data/ or Drive root."
    print("bootstrap OK · OUT_DIR =", os.environ["PICKO_OUT_DIR"])
else:
    print("Not on Colab — running locally (CPU).")

## 1 · Setup & data overview

In [ ]:
# ensure the repo root is importable (works from notebooks/research/, Colab, etc.)
import os, sys
_here = os.path.abspath(os.getcwd())
for _ in range(6):
    if os.path.exists(os.path.join(_here, "scripts", "picko_research.py")): break
    _here = os.path.dirname(_here)
if os.path.isdir("/content/picko"): _here = "/content/picko"
if _here not in sys.path: sys.path.insert(0, _here)

from scripts.picko_research import *
import pandas as pd, numpy as np, matplotlib.pyplot as plt
try:
    import seaborn as sns; sns.set_theme(style="whitegrid")
except Exception:
    sns = None
from tqdm.auto import tqdm

cat, tok, raw, FOCUS, OUT_DIR = load_context()

### The 40 focus tools\nOne row per tool, with its family, category and **parameter count / bucket**.

In [ ]:
display(tools_dataframe(cat, FOCUS))

### All examples for these 40 tools\nOne row per training example (query → gold tool), tagged with the gold tool's **param bucket**.

In [ ]:
ex_df = examples_dataframe(cat, raw, FOCUS)
print("examples:", ex_df.shape[0], "| per param bucket:", ex_df["param_bucket"].value_counts().to_dict())
display(ex_df.head(10))

## 2 · Configure the repeated sampling\nEach iteration draws `TOOLS_PER_BUCKET` tools from **each** bucket (0 / 1 / 2-3 / 4+) into one small model.

In [ ]:
N_ITER          = 5     # <- number of independent (tool-sample + finetune) iterations
TOOLS_PER_BUCKET = 2     # tools drawn from each param bucket per iteration
CAP_PER_TOOL     = 40
EPOCHS           = 1
EVAL_SUBSAMPLE   = 40
RUN_TRAIN        = True
FORCE_RETRAIN    = False
print("param buckets available:", tools_dataframe(cat, FOCUS)["param_bucket"].value_counts().to_dict())

## 3 · Run the iterations

In [ ]:
rows = []
for i in range(N_ITER):
    names = sample_stratified(cat, FOCUS, TOOLS_PER_BUCKET, seed=i)
    R = finetune_and_eval(cat, raw, tok, names, f"depth_iter{i}", OUT_DIR,
                          cap=CAP_PER_TOOL, epochs=EPOCHS, compact=False, token_aware=True,
                          eval_subsample=EVAL_SUBSAMPLE, run_train=RUN_TRAIN, force_retrain=FORCE_RETRAIN)
    for tool, s in R["metrics"]["per_tool"].items():
        _, tot = cat.params_of(tool)
        rows.append({"iteration": i, "tool": tool, "total_params": tot,
                     "param_bucket": param_bucket(tot), "n": s["n"],
                     "selection_acc": s["selection_acc"],
                     "args_exact_acc": s["args_exact_acc"], "param_f1": s["param_f1"]})
    print(f"  iter {i}: tools={names}")
depth = pd.DataFrame(rows)
import json as _json
_json.dump(rows, open(os.path.join(OUT_DIR, "depth_results.json"), "w"), indent=2)
print("collected", len(depth), "per-tool measurements across", N_ITER, "iterations")
display(depth.head(12))

## 4 · Extraction accuracy per parameter bucket (mean ± std)

In [ ]:
# per-iteration bucket means first (paired within iteration), then mean/std across iterations
per_iter = (depth.groupby(["iteration","param_bucket"])[["args_exact_acc","param_f1"]]
            .mean().reset_index())
agg = (per_iter.groupby("param_bucket")
       .agg(args_mean=("args_exact_acc","mean"), args_std=("args_exact_acc","std"),
            pf1_mean=("param_f1","mean"), pf1_std=("param_f1","std"),
            n_iter=("iteration","nunique"))
       .reindex([b for b in PARAM_BUCKET_ORDER if b in per_iter["param_bucket"].values]))
display(agg.round(3))

x = np.arange(len(agg)); w = 0.38
fig, ax = plt.subplots(figsize=(8,4.5))
ax.bar(x-w/2, agg["args_mean"], w, yerr=agg["args_std"].fillna(0), capsize=4, color="#4C72B0", label="args_exact_acc")
ax.bar(x+w/2, agg["pf1_mean"], w, yerr=agg["pf1_std"].fillna(0), capsize=4, color="#DD8452", label="param_f1")
# overlay each iteration's bucket mean as points
for _, r in per_iter.iterrows():
    xi = list(agg.index).index(r["param_bucket"]) if r["param_bucket"] in list(agg.index) else None
    if xi is not None: ax.scatter(xi-w/2, r["args_exact_acc"], color="#243b57", s=14, zorder=3)
ax.set_xticks(x); ax.set_xticklabels(agg.index); ax.set_ylim(0,1)
ax.set_xlabel("# parameters (bucket)"); ax.set_ylabel("accuracy")
ax.set_title(f"Depth: parameter extraction vs #params ({agg['n_iter'].max()} iterations)"); ax.legend()
plt.tight_layout(); plt.show()

## 5 · Per-tool scatter (all iterations)

In [ ]:
tool_mean = depth.groupby(["tool","total_params"])["args_exact_acc"].mean().reset_index()
plt.figure(figsize=(7.5,4.5))
plt.scatter(tool_mean["total_params"], tool_mean["args_exact_acc"], s=55, color="#4C72B0")
for _, r in tool_mean.iterrows():
    plt.annotate(r["tool"].split("_")[0], (r["total_params"], r["args_exact_acc"]), fontsize=7)
plt.xlabel("# parameters in tool"); plt.ylabel("mean args_exact_acc")
plt.title("Depth: per-tool extraction vs parameter count"); plt.tight_layout(); plt.show()

## 6 · Read-out

Argument extraction is near-solved for **0–1 parameter** tools and **degrades for multi-parameter (4+)**
tools — the error bars show it's a consistent effect across independent tool samples, not one unlucky
model. This is where a small specialist model needs the most help (and where finetuning gains most).